# HW5 Mason Holcombe

In [1]:
import gensim
import gensim.downloader as api

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
np.random.seed(2026)


from HW5_Utilities import load_movie_polarity_reviews, pretrained_model_similarity, sentence_to_avg

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from sklearn.metrics import accuracy_score

# 1)

In [26]:
EMBEDDING_DIM = 300
WINDOW_SIZE = 3

dataset = api.load("text8")
model = gensim.models.Word2Vec(sentences=dataset,
                               vector_size=EMBEDDING_DIM,
                               window=WINDOW_SIZE)

print(f"STEP 1 | Completed {EMBEDDING_DIM} dim training.")

STEP 1 | Completed 300 dim training.


In [27]:
similar_words = [
    ("king", "queen"),
    ("fast", "quick"),
    ("begin", "start"),
    ("ocean", "sea"),
    ("hot", "warm")
]

print(f"STEP 1 | Similarity Scores:")
for w1, w2 in similar_words:
    wv1 = model.wv[w1].reshape(1, -1)
    wv2 = model.wv[w2].reshape(1, -1)

    sim_score = cosine_similarity(wv1, wv2).item()

    print(f"\t({w1}, {w2}) -> {sim_score:.4f}")

STEP 1 | Similarity Scores:
	(king, queen) -> 0.6208
	(fast, quick) -> 0.6405
	(begin, start) -> 0.6801
	(ocean, sea) -> 0.5734
	(hot, warm) -> 0.7437


# 2)

In [28]:
print(f"STEP 2 | Downloading google-news-300...")
pretrained_model = api.load('word2vec-google-news-300')
print(f"STEP 2 | Download complete.")

STEP 2 | Downloading google-news-300...
STEP 2 | Download complete.


In [29]:
words = ["neural", "calculus", "happiness", "flower"]

print(f"STEP 2 | Large model qualitative behavior:")
for word in words:
    top5 = pretrained_model.most_similar(word, topn=5)
    
    print(f"\nTop 5 words similar to '{word}':")
    for i, (w, score) in enumerate(top5, start=1):
        print(f"{i}. {w:^18} {score:.4f}")


STEP 2 | Large model qualitative behavior:

Top 5 words similar to 'neural':
1.      neuronal      0.7805
2.      neurons       0.7326
3.  neural_circuits   0.7253
4.       neuron       0.7174
5.      cortical      0.6941

Top 5 words similar to 'calculus':
1.      algebra       0.6411
2.      Calculus      0.5957
3.        math        0.5777
4.    trigonometry    0.5716
5.    precalculus     0.5526

Top 5 words similar to 'happiness':
1.    contentment     0.7695
2.        joy         0.6183
3.     Happiness      0.6116
4.      hapiness      0.5749
5.   contentedness    0.5575

Top 5 words similar to 'flower':
1.       floral       0.7494
2.      flowers       0.7489
3.       roses        0.6977
4.       orchid       0.6929
5.       tulip        0.6629


# 3)

In [30]:
wordsim353 = pd.read_csv("wordsim_similarity_goldstandard.csv")
wordsim353.columns = ["word1", "word2", "mean_human_score"]

In [31]:
wordsim353["model_score"] = wordsim353.apply(lambda row: pretrained_model_similarity(pretrained_model, row["word1"], row["word2"]), axis=1)

In [32]:
rank_coef = spearmanr(wordsim353["mean_human_score"], wordsim353["model_score"]).statistic.item()
print(f"STEP 3 | Spearman's Rank Correlation Coefficient on WordSim-353: {rank_coef:.4f}")

STEP 3 | Spearman's Rank Correlation Coefficient on WordSim-353: 0.7000


# 4)

In [10]:
# run - running + swimming = swim
pretrained_model.most_similar(
    positive=["run", "swimming"],
    negative=["running"],
    topn=5
)

[('swim', 0.6857838034629822),
 ('swims', 0.5815677642822266),
 ('swimmers', 0.5586199760437012),
 ('Swimming', 0.5309520959854126),
 ('swimmer', 0.5201396346092224)]

In [11]:
# hot - temperature + emotion = unbridled_passion, passion, excitment
pretrained_model.most_similar(
    positive=["hot", "emotion"],
    negative=["temperature"],
    topn=5
)

[('emotions', 0.4435824155807495),
 ('unbridled_passion', 0.42061567306518555),
 ('passion', 0.40441808104515076),
 ('excitement', 0.4009501338005066),
 ('angst', 0.38935503363609314)]

In [12]:
# einstein - physics + music = jonas_brothers, eminem, mariah_carey
pretrained_model.most_similar(
    positive=["einstein", "music"],
    negative=["physics"],
    topn=5
)

[('jonas_brothers', 0.539109468460083),
 ('choons', 0.5335083603858948),
 ('eminem', 0.5320717096328735),
 ('listenin', 0.5252184867858887),
 ('mariah_carey', 0.5245700478553772)]

# 5)

In [34]:
embedding_dim = pretrained_model.vector_size

In [ ]:
all_reviews = load_movie_polarity_reviews()

all_reviews_dict = dict(sorted(all_reviews.items(), key=lambda item: np.random.rand()))

# Split into T/D/Test: 70/15/15
train_reviews = dict(list(all_reviews_dict.items())[:int(len(all_reviews_dict) * 0.7)])
dev_reviews = dict(list(all_reviews_dict.items())[int(len(all_reviews_dict) * 0.7):int(len(all_reviews_dict) * 0.85)])
test_reviews = dict(list(all_reviews_dict.items())[int(len(all_reviews_dict) * 0.85):])

In [ ]:
train_X = np.array([sentence_to_avg(s, model=pretrained_model, embedding_dim=embedding_dim) for s in train_reviews])
train_y = np.array(list(train_reviews.values()))

dev_X = np.array([sentence_to_avg(s, model=pretrained_model, embedding_dim=embedding_dim) for s in dev_reviews])
dev_y = np.array(list(dev_reviews.values()))

test_X = np.array([sentence_to_avg(s, model=pretrained_model, embedding_dim=embedding_dim) for s in test_reviews])
test_y = np.array(list(test_reviews.values()))

In [48]:
input_dim = embedding_dim

model = Sequential([
    Dense(256, activation='relu', input_shape=(input_dim,)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # binary classification
])
model.compile(
    optimizer='RMSprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.fit(
    train_X, train_y,
    epochs=7,
    batch_size=8,
    validation_data=(dev_X, dev_y)
)

Epoch 1/7


c:\Users\mason\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


933/933 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7262 - loss: 0.5386 - val_accuracy: 0.7817 - val_loss: 0.4652
Epoch 2/7
933/933 ━━━━━━━━━━━━━━━━━━━━ 1s 949us/step - accuracy: 0.7634 - loss: 0.4881 - val_accuracy: 0.7842 - val_loss: 0.4533
Epoch 3/7
933/933 ━━━━━━━━━━━━━━━━━━━━ 1s 938us/step - accuracy: 0.7762 - loss: 0.4708 - val_accuracy: 0.7917 - val_loss: 0.4634
Epoch 4/7
933/933 ━━━━━━━━━━━━━━━━━━━━ 1s 948us/step - accuracy: 0.7761 - loss: 0.4593 - val_accuracy: 0.7717 - val_loss: 0.4677
Epoch 5/7
933/933 ━━━━━━━━━━━━━━━━━━━━ 1s 944us/step - accuracy: 0.7915 - loss: 0.4480 - val_accuracy: 0.7630 - val_loss: 0.5129
Epoch 6/7
933/933 ━━━━━━━━━━━━━━━━━━━━ 1s 931us/step - accuracy: 0.7896 - loss: 0.4433 - val_accuracy: 0.7880 - val_loss: 0.4618
Epoch 7/7
933/933 ━━━━━━━━━━━━━━━━━━━━ 1s 932us/step - accuracy: 0.7962 - loss: 0.4326 - val_accuracy: 0.7842 - val_loss: 0.4784


In [49]:
y_pred_probs = model.predict(test_X)
y_pred = (y_pred_probs > 0.5).astype(int)

from sklearn.metrics import accuracy_score

print("Test accuracy:", accuracy_score(test_y, y_pred))

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 566us/step
Test accuracy: 0.775
